# Traductor de txt a SQL

**Índice**   
1. [Imports](#imports)
2. [Cargamos los modelos](#cargamos-los-modelos)
3. [Creamos la base de datos](#creamos-la-base-de-datos)
4. [Mapeo de columnas](#mapeo-de-columnas)
5. [Sinonimos/palabras clave (ES/EN)](#sinonimos--palabras-clave-esen)
6. [Metricas](#metricas)
7. [Agrupaciones temporales](#agrupaciones-temporales)
8. [Funcion que detecta el idioma](#funcion-que-detecta-el-idioma) .
9. [Filtro de fecha](#filtro-de-fecha)
10. [Filtro minus y mayus](#filtro-minus-y-mayus)
11. [Filtro agrupaciones](#filtro-agrupaciones)
12. [Detección de where](#detección-de-where).
13. [Detección de group](#detección-de-group).
14. [Detección de filtro](#detección-de-filtro).
15. [Generador de SQL](#generador-de-SQL)
16. [Pruebas](#pruebas)

## Imports

In [59]:
import re
import spacy
from langdetect import detect
import pandas as pd

## Cargamos los modelos 

In [ ]:
# Modelos
nlp_es = spacy.load("es_core_news_sm") # Español
nlp_en = spacy.load("en_core_web_sm") # English

## Creamos la base de datos

In [61]:
# Cargar los CSVs (ajusta las rutas a tus archivos locales)
clientes = pd.read_csv("./data/clientes_ecommerce.csv")
transacciones = pd.read_csv("./data/transacciones_ecommerce.csv")

In [62]:
df = pd.merge(transacciones, clientes, on="id_cliente", how="inner")
TABLE_NAME = "dataset_merged"
# IMPORTANTE: en tu df mergeado las columnas son las del CSV, aquí asumo que usas las españolas.

In [78]:
df

,id_transaccion,id_cliente,fecha_compra,producto,categoria_producto,precio_unitario,cantidad,importe_total,metodo_pago,coste_envio,coste_fabricacion,nombre,apellidos,email,pais,ciudad,edad,genero
0,1,3849,2024-11-02,Powerbank Anker 20000 mAh,Accesorios,118.33,3,354.99,paypal,3.07,40.39,Laura,Fernández Álvarez,laura.fernandez@gmail.com,Portugal,Lisboa,36,M
1,2,2704,2023-09-01,Garmin Forerunner 255,Relojes inteligentes,659.40,2,1318.80,tarjeta,9.53,391.72,Laura,Torres Martínez,laura.torres1@icloud.com,España,Madrid,52,M
2,3,447,2024-05-24,Huawei Watch GT 4,Relojes inteligentes,150.78,1,150.78,paypal,9.35,72.56,Elena,Fernández Ruiz,elena.fernandez1@live.com,Argentina,Buenos Aires,36,M
3,4,790,2023-08-18,Samsung Galaxy S23,Móviles,1199.81,2,2399.62,bizum,12.44,619.11,María,Pérez Vázquez,maria.perez@live.com,Argentina,Buenos Aires,34,F
4,5,4540,2024-12-12,Powerbank Anker 20000 mAh,Accesorios,175.10,2,350.20,bizum,4.41,38.44,Cristina,Vázquez Fernández,cristina.vazquez2@hotmail.com,España,Barcelona,34,Otro/NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,14996,2493,2024-02-08,HP Spectre x360,Portátiles,921.64,2,1843.28,paypal,22.01,699.77,Marta,Domínguez Gil,marta.dominguez@hotmail.com,Reino Unido,Manchester,36,F
14996,14997,1591,2023-05-20,Google Pixel 8,Móviles,623.51,1,623.51,tarjeta,10.22,350.31,Elena,Pérez Muñoz,elena.perez1@yahoo.com,Portugal,Lisboa,48,Otro/NA
14997,14998,2963,2024-02-17,Fitbit Versa 4,Relojes inteligentes,659.78,2,1319.56,bizum,7.37,276.57,Laura,Torres Romero,laura.torres2@icloud.com,España,Madrid,31,M
14998,14999,1270,2023-08-11,Huawei Watch GT 4,Relojes inteligentes,159.61,1,159.61,bizum,12.21,67.48,Pablo,Jiménez Pérez,pablo.jimenez@gmail.com,España,Barcelona,24,F


## Mapeo de columnas

In [64]:
COLS = {
    "id_cliente","nombre","apellidos","email","pais","ciudad","edad","genero",
    "id_transaccion","fecha_compra","producto","categoria_producto",
    "precio_unitario","cantidad","importe_total","metodo_pago",
    "coste_envio","coste_fabricacion"
}

## Sinonimos / palabras clave (ES/EN)

In [65]:
# Sinónimos / palabras clave -> columnas (ES/EN)
SYN_TO_COL = {
    "es": {
        # métricas
        "ventas": "importe_total",
        "ingresos": "importe_total",
        "beneficios": "importe_total",
        "facturacion": "importe_total",
        "importe": "importe_total",
        "total": "importe_total",
        "unidades": "cantidad",
        "cantidad": "cantidad",
        "precio": "precio_unitario",
        "envio": "coste_envio",
        "fabricacion": "coste_fabricacion",
        # dimensiones
        "pais": "pais",
        "ciudad": "ciudad",
        "producto": "producto",
        "categoria": "categoria_producto",
        "genero": "genero",
        "edad": "edad",
        "metodo": "metodo_pago",
        "pago": "metodo_pago",
        "fecha": "fecha_compra",
        "compra": "fecha_compra",
        "transaccion": "id_transaccion",
        "cliente": "id_cliente",
    },
    "en": {
        # si el usuario pregunta en inglés, seguimos generando SQL con columnas ES
        # (porque tu dataset está en ES). Solo traducimos la intención.
        "sales": "importe_total",
        "revenue": "importe_total",
        "amount": "importe_total",
        "total": "importe_total",
        "units": "cantidad",
        "quantity": "cantidad",
        "price": "precio_unitario",
        "shipping": "coste_envio",
        "manufacturing": "coste_fabricacion",
        # dimensiones
        "country": "pais",
        "city": "ciudad",
        "product": "producto",
        "category": "categoria_producto",
        "gender": "genero",
        "age": "edad",
        "payment": "metodo_pago",
        "date": "fecha_compra",
        "purchase": "fecha_compra",
        "transaction": "id_transaccion",
        "client": "id_cliente",
        "customer": "id_cliente",
    }
}

## Metricas

### METER MÁS FUNCIONES (MAX, MIN)

In [66]:
AGG_WORDS = {
    "es": {
        "avg": {"promedio", "media", "promediar"},
        "sum": {"suma", "total", "sumar"},
        "count": {"cuantos", "cuántos", "numero", "número", "conteo", "contar", "Cuántas"},
    },
    "en": {
        "avg": {"average", "avg", "mean"},
        "sum": {"sum", "total"},
        "count": {"count", "how", "many", "number"},
    }
}

## Agrupaciones temporales

In [67]:
TIME_GROUP_WORDS = {
    "es": {
        "quarter": {"trimestre", "trimestral"},
        "month": {"mes", "mensual"},
        "year": {"año", "anual"},
    },
    "en": {
        "quarter": {"quarter", "qtr"},
        "month": {"month", "monthly"},
        "year": {"year", "yearly", "annual"},
    }
}

## Funcion que detecta el idioma

In [68]:
def detectar_idioma(texto: str):
    lang = detect(texto)
    if lang == "es":
        return nlp_es(texto), "es"
    elif lang == "en":
        return nlp_en(texto), "en"
    else:
        raise ValueError(f"Idioma no soportado: {lang}")

## Filtro de fecha

In [69]:
def _find_years(texto: str):
    # años 1900-2099 CÓDIGO
    # años 2023-2024 REALIDAD
    return sorted(set(re.findall(r"\b(19\d{2}|20\d{2})\b", texto)))

## Prepocesado minus y mayus

In [70]:
def _normalize_tokens(doc):
    # lemmas en minúscula, sin puntuación/espacios
    return [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space]

## Filtro agrupaciones

In [71]:
def detectar_agregacion(tokens, idioma):
    # default: None (si no pide nada, se puede devolver *)
    for agg, words in AGG_WORDS[idioma].items():
        if any(w in tokens for w in words):
            return agg
    return None

## Detección de where

In [72]:
def detectar_metricas(tokens, idioma):
    # Busca la primera métrica "razonable"
    # Si menciona ventas/importe -> importe_total; unidades -> cantidad; etc.
    for tok in tokens:
        if tok in SYN_TO_COL[idioma]:
            col = SYN_TO_COL[idioma][tok]
            if col in {"importe_total", "cantidad", "precio_unitario", "coste_envio", "coste_fabricacion"}:
                return col
    # fallback: si habla de promedio sin métrica explícita, asumimos ventas
    return None

## Detección de group

In [73]:
def detectar_groupbys(tokens, idioma):
    group_cols = []
    # Tiempo
    if any(w in tokens for w in TIME_GROUP_WORDS[idioma]["quarter"]):
        group_cols.append("QUARTER(fecha_compra)")
        group_cols.append("YEAR(fecha_compra)")
    elif any(w in tokens for w in TIME_GROUP_WORDS[idioma]["month"]):
        group_cols.append("MONTH(fecha_compra)")
        group_cols.append("YEAR(fecha_compra)")
    elif any(w in tokens for w in TIME_GROUP_WORDS[idioma]["year"]):
        group_cols.append("YEAR(fecha_compra)")
    # Dimensiones típicas si el usuario las menciona
    dims_priority = ["pais", "ciudad", "categoria_producto", "producto", "genero", "metodo_pago"]
    mentioned = set()
    for tok in tokens:
        if tok in SYN_TO_COL[idioma]:
            mentioned.add(SYN_TO_COL[idioma][tok])
    for d in dims_priority:
        if d in mentioned and d in COLS:
            group_cols.append(d)
    # Elimina duplicados manteniendo orden
    seen = set()
    out = []
    for g in group_cols:
        if g not in seen:
            out.append(g)
            seen.add(g)
    return out

## Detección de filtro

In [74]:
def detectar_filtros(doc, tokens, idioma):
    where = []
    # Año(s)
    years = _find_years(doc.text)
    if years:
        # Si hay varios, usamos IN
        if len(years) == 1:
            where.append(f"YEAR(fecha_compra) = {years[0]}")
        else:
            years_list = ",".join(years)
            where.append(f"YEAR(fecha_compra) IN ({years_list})")
    # País / ciudad vía entidades LOC/GPE
    for ent in doc.ents:
        if ent.label_ in {"LOC", "GPE"}:
            # Heurística: si menciona "ciudad" cerca, filtra por ciudad; si no, por país.
            txt = ent.text.replace("‘", "‘’")
            # Ventana simple alrededor de la entidad
            span_start = max(ent.start - 2, 0)
            span_end = min(ent.end + 2, len(doc))
            window = " ".join([t.lemma_.lower() for t in doc[span_start:span_end]])
            if ("ciudad" in window) or ("city" in window):
                where.append(f"ciudad = ‘{txt}‘")
            else:
                where.append(f"pais = ‘{txt}‘")
    # Producto / categoría por patrón "producto X" / "categoría Y"
    text_lower = doc.text.lower()
    # ES: "producto iphone", "categoría electronica"
    m_prod = re.search(r"(producto)\s+([a-z0-9_\-áéíóúñ ]{2,})", text_lower)
    if m_prod:
        val = m_prod.group(2).strip()
        # corta si aparecen conectores comunes
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        where.append(f"producto LIKE ‘%{val.replace('\'','\'\'')}%")
    m_cat = re.search(r"(categor[ií]a)\s+([a-z0-9_\-áéíóúñ ]{2,})", text_lower)
    if m_cat:
        val = m_cat.group(2).strip()
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        where.append(f"categoria_producto LIKE ‘%{val.replace('\'','\'\'')}%’")
    # Género simple (M/F, masculino/femenino, male/female)
    if re.search(r"\b(masculino|hombre|male|m)\b", text_lower):
        where.append("genero IN (‘M’,‘Masculino’,‘male’,‘Male’)")
    if re.search(r"\b(femenino|mujer|female|f)\b", text_lower):
        where.append("genero IN (‘F’,‘Femenino’,‘female’,‘Female’)")
    # Edad: "mayores de 30", "menores de 25"
    m_gt = re.search(r"(mayores de|más de|over|older than)\s+(\d{1,3})", text_lower)
    if m_gt:
        where.append(f"edad > {int(m_gt.group(2))}")
    m_lt = re.search(r"(menores de|menos de|under|younger than)\s+(\d{1,3})", text_lower)
    if m_lt:
        where.append(f"edad < {int(m_lt.group(2))}")
    # Dedup
    where_out = []
    seen = set()
    for w in where:
        if w not in seen:
            where_out.append(w)
            seen.add(w)
    return where_out

## Generador de SQL

In [75]:
def generar_sql(texto: str):
    doc, idioma = detectar_idioma(texto)
    tokens = _normalize_tokens(doc)
    agg = detectar_agregacion(tokens, idioma)
    metric = detectar_metricas(tokens, idioma)
    # Defaults "razonables"
    if agg in {"avg", "sum"} and metric is None:
        metric = "importe_total"
    if agg is None and metric is not None:
        # si menciona métrica pero no agg, asumimos SUM para ventas/unidades
        if metric in {"importe_total", "cantidad"}:
            agg = "sum"
    group_by = detectar_groupbys(tokens, idioma)
    where = detectar_filtros(doc, tokens, idioma)
    # SELECT
    select_parts = []
    if group_by:
        select_parts.extend(group_by)
    if agg == "avg":
        select_parts.append(f"AVG({metric}) AS promedio_{metric}")
    elif agg == "sum":
        select_parts.append(f"SUM({metric}) AS total_{metric}")
    elif agg == "count":
        # Si pide conteo, contamos transacciones por defecto
        select_parts.append("COUNT(id_transaccion) AS conteo_transacciones")
    else:
        # Sin intención: devuelve columnas principales (evita SELECT *)
        select_parts.append("id_transaccion")
        select_parts.append("fecha_compra")
        select_parts.append("importe_total")
        select_parts.append("cantidad")
    sql = "SELECT " + ", ".join(select_parts) + f" FROM {TABLE_NAME}"
    if where:
        sql += " WHERE " + " AND ".join(where)
    if group_by:
        sql += " GROUP BY " + ", ".join(group_by)
    sql += ";"
    return sql

## Pruebas

In [85]:
print(generar_sql("¿Cuántas transacciones hubo en 2024?"))

SELECT COUNT(id_transaccion) AS conteo_transacciones FROM dataset_merged WHERE YEAR(fecha_compra) = 2024;


In [ ]:
print(generar_sql("¿Cuántas transacciones hubo en España en el año 2023?"))

SELECT YEAR(fecha_compra), COUNT(id_transaccion) AS conteo_transacciones FROM dataset_merged WHERE YEAR(fecha_compra) = 2023 AND pais = ‘España‘ GROUP BY YEAR(fecha_compra);


In [79]:
print(generar_sql("Ventas máximas por país en enero"))

SELECT id_transaccion, fecha_compra, importe_total, cantidad FROM dataset_merged;


In [80]:
print(generar_sql("Cuántas transacciones en España en 2023 por mes"))

SELECT MONTH(fecha_compra), YEAR(fecha_compra), COUNT(id_transaccion) AS conteo_transacciones FROM dataset_merged WHERE YEAR(fecha_compra) = 2023 AND pais = ‘España‘ GROUP BY MONTH(fecha_compra), YEAR(fecha_compra);


In [81]:
print(generar_sql("Total sales in 2024 by country"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM dataset_merged WHERE YEAR(fecha_compra) = 2024 GROUP BY pais;


In [82]:
print(generar_sql("Muéstrame el promedio de ventas en 2023 por trimestre"))

SELECT QUARTER(fecha_compra), YEAR(fecha_compra), AVG(importe_total) AS promedio_importe_total FROM dataset_merged WHERE YEAR(fecha_compra) = 2023 GROUP BY QUARTER(fecha_compra), YEAR(fecha_compra);


In [84]:
print(generar_sql("Número de transacciones en México en 2024"))

SELECT COUNT(id_transaccion) AS conteo_transacciones FROM dataset_merged WHERE YEAR(fecha_compra) = 2024 AND pais = ‘México‘;


In [83]:
print(generar_sql("¿Cuántas transacciones hubo en España en el año 2023?"))

SELECT YEAR(fecha_compra), COUNT(id_transaccion) AS conteo_transacciones FROM dataset_merged WHERE YEAR(fecha_compra) = 2023 AND pais = ‘España‘ GROUP BY YEAR(fecha_compra);
